# 08 — MatterGen + MatterSim Convex Hull Pipeline

End-to-end pipeline for discovering thermodynamically stable crystal structures
in a binary chemical system:

1. **MatterGen** generates candidate structures conditioned on the chemical system
   (1000-step diffusion denoising on GPU)
2. **MatterSim** relaxes all candidates and predicts their energies
3. Formation energies and the **convex hull** are computed self-consistently

### Setup

```bash
# Core project
pip install -e ".[notebook]"

# MatterGen + MatterSim (--no-deps to avoid torch version conflicts)
pip install mattergen mattersim --no-deps

# Required transitive dependencies
pip install hydra-core omegaconf pytorch-lightning huggingface_hub loguru seekpath torch-runstats "e3nn>=0.5"

# PyG + extensions (match your torch+CUDA version)
pip install torch-geometric
pip install torch-scatter torch-cluster -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
```

#### JIT compatibility patches (torch 2.11)

Three `@torch.jit.script` decorators in helper libraries segfault on torch 2.11.
They have been removed in the installed packages (one-time fix):

| File | Decorator removed |
|---|---|
| `torch_sparse/storage.py:21` | `@torch.jit.script` on `SparseStorage` |
| `torch_sparse/tensor.py:12` | `@torch.jit.script` on `SparseTensor` |
| `torch_runstats/scatter.py:28,53,92` | `@torch.jit.script` on 3 scatter functions |

Neither MatterGen (GemNetT) nor MatterSim (M3GNet) use TorchScript for inference,
so removing these decorators has no effect on model performance.

Hardware: **NVIDIA GPU strongly recommended** (both models load onto CUDA).

In [4]:
# ── Cell 1: Configuration ──────────────────────────────────────────────

ELEMENT_A = "Li"
ELEMENT_B = "O"
N_CANDIDATES = 32          # structures to generate with MatterGen
E_ABOVE_HULL_TARGET = 0.0  # condition MatterGen toward stable structures
GUIDANCE_FACTOR = 2.0      # classifier-free guidance strength

# Derived
SYSTEM = f"{ELEMENT_A}-{ELEMENT_B}"

# Paths
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "mattergen" / SYSTEM
REF_CACHE = PROJECT_ROOT / "data" / "mattersim_references.yaml"
RESULTS_CSV = PROJECT_ROOT / "data" / "results" / f"{SYSTEM}_mattersim_hull.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

# GPU detection
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"System:     {SYSTEM}")
print(f"Candidates: {N_CANDIDATES}")
print(f"Device:     {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — generation and relaxation will be very slow.")

System:     Li-O
Candidates: 32
Device:     cuda
GPU:        NVIDIA RTX 500 Ada Generation Laptop GPU


In [5]:
# ── Cell 2: MatterGen — generate candidate structures ─────────────────
#
# MatterGen is a diffusion model that generates crystal structures by
# denoising random noise over 1000 steps (fixed by the D3PM schedule).
# With classifier-free guidance it runs ~2 forward passes per step through
# GemNetT on GPU.  Expect ~9 min for a batch of 32 structures.

from mattergen.generator import CrystalGenerator
from mattergen.common.utils.data_classes import MatterGenCheckpointInfo

checkpoint_info = MatterGenCheckpointInfo.from_hf_hub(
    "chemical_system_energy_above_hull",
)

generator = CrystalGenerator(
    checkpoint_info=checkpoint_info,
    properties_to_condition_on={
        "chemical_system": SYSTEM,
        "energy_above_hull": E_ABOVE_HULL_TARGET,
    },
    batch_size=N_CANDIDATES,
    num_batches=1,
    diffusion_guidance_factor=GUIDANCE_FACTOR,
)

structures = generator.generate(output_dir=OUTPUT_DIR)
print(f"\nGenerated {len(structures)} candidate structures for {SYSTEM}")
for i, s in enumerate(structures[:8]):
    print(f"  [{i}] {s.composition.reduced_formula:>10s}  "
          f"{s.num_sites} atoms  "
          f"vol={s.volume:.1f} A³")
if len(structures) > 8:
    print(f"  ... ({len(structures) - 8} more)")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mattergen/resolve/main/checkpoints/chemical_system_energy_above_hull/checkpoints/last.ckpt "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mattergen/resolve/main/checkpoints/chemical_system_energy_above_hull/config.yaml "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/mattergen/ea430eab64b80855029c2941b9fda15f245a771a/checkpoints%2Fchemical_system_energy_above_hull%2Fconfig.yaml "HTTP/1.1 200 OK"



Model config:
adapter:
  adapter:
    _target_: mattergen.adapter.GemNetTAdapter
    atom_type_diffusion: mask
    denoise_atom_types: true
    gemnet:
      _target_: mattergen.common.gemnet.gemnet_ctrl.GemNetTCtrl
      atom_embedding:
        _target_: mattergen.common.gemnet.layers.embedding_block.AtomEmbedding
        emb_size: 512
        with_mask_type: true
      condition_on_adapt:
      - chemical_system
      - energy_above_hull
      cutoff: 7.0
      emb_size_atom: 512
      emb_size_edge: 512
      latent_dim: 512
      max_cell_images_per_dim: 5
      max_neighbors: 50
      num_blocks: 4
      num_targets: 1
      otf_graph: true
      regress_stress: true
      scale_file: /scratch/amlt_code/mattergen/common/gemnet/gemnet-dT.json
    hidden_dim: 512
    property_embeddings: {}
    property_embeddings_adapt:
      chemical_system:
        _target_: mattergen.property_embeddings.PropertyEmbedding
        conditional_embedding_module:
          _target_: mattergen.proper

INFO:mattergen.common.utils.eval_utils:Loading model from checkpoint: /home/hhoechter/.cache/huggingface/hub/models--microsoft--mattergen/snapshots/ea430eab64b80855029c2941b9fda15f245a771a/checkpoints/chemical_system_energy_above_hull/checkpoints/last.ckpt



Sampling config:
sampler_partial:
  _target_: mattergen.diffusion.sampling.classifier_free_guidance.GuidedPredictorCorrector.from_pl_module
  'N': 1000
  eps_t: 0.001
  _partial_: true
  guidance_scale: 2.0
  remove_conditioning_fn:
    _target_: mattergen.property_embeddings.SetUnconditionalEmbeddingType
  keep_conditioning_fn:
    _target_: mattergen.property_embeddings.SetConditionalEmbeddingType
  predictor_partials:
    pos:
      _target_: mattergen.diffusion.wrapped.wrapped_predictors_correctors.WrappedAncestralSamplingPredictor
      _partial_: true
    cell:
      _target_: mattergen.common.diffusion.predictors_correctors.LatticeAncestralSamplingPredictor
      _partial_: true
    atomic_numbers:
      _target_: mattergen.diffusion.d3pm.d3pm_predictors_correctors.D3PMAncestralSamplingPredictor
      predict_x0: true
      _partial_: true
  corrector_partials:
    pos:
      _target_: mattergen.diffusion.wrapped.wrapped_predictors_correctors.WrappedLangevinCorrector
      _par

Generating samples:   0%|          | 0/1 [09:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ── Cell 3: MatterSim — compute elemental reference energies ──────────
#
# For formation energy we need the per-atom energy of each pure element
# in its ground-state crystal structure, relaxed with MatterSim.
# Results are cached to disk so this only runs once per element.

from hullgap.calculators import get_calculator
from hullgap.references import get_elemental_references

import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

calc = get_calculator("mattersim", device=DEVICE)

refs = get_elemental_references(
    elements=[ELEMENT_A, ELEMENT_B],
    calculator=calc,
    cache_path=REF_CACHE,
)

for el, e in refs.items():
    print(f"  {el}: {e:.6f} eV/atom")

In [ ]:
# ── Cell 4: MatterSim — relax all candidate structures ────────────────

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pymatgen.io.ase import AseAtomsAdaptor
from ase.filters import FrechetCellFilter
from ase.optimize import LBFGS

results = []

for i, struct in enumerate(tqdm(structures, desc="Relaxing candidates")):
    atoms = AseAtomsAdaptor.get_atoms(struct)
    atoms.calc = calc

    try:
        opt_target = FrechetCellFilter(atoms)
        opt = LBFGS(opt_target, logfile=None)
        opt.run(fmax=0.02, steps=500)

        e_total = float(atoms.get_potential_energy())
        n_atoms = len(atoms)
        forces = atoms.get_forces()
        fmax = float(np.max(np.linalg.norm(forces, axis=1)))

        results.append({
            "idx": i,
            "formula": struct.composition.reduced_formula,
            "n_atoms": n_atoms,
            "e_total_eV": e_total,
            "e_per_atom_eV": e_total / n_atoms,
            "fmax_eV_A": fmax,
            "volume_A3": atoms.get_volume(),
            "status": "converged" if fmax < 0.02 else "max_steps",
        })
    except Exception as exc:
        results.append({
            "idx": i,
            "formula": struct.composition.reduced_formula,
            "n_atoms": struct.num_sites,
            "e_total_eV": np.nan,
            "e_per_atom_eV": np.nan,
            "fmax_eV_A": np.nan,
            "volume_A3": np.nan,
            "status": f"failed: {exc}",
        })

df = pd.DataFrame(results)
print(f"\nRelaxed {len(df)} structures")
print(f"  converged: {(df.status == 'converged').sum()}")
print(f"  max_steps: {(df.status == 'max_steps').sum()}")
print(f"  failed:    {df.status.str.startswith('failed').sum()}")
df.head(10)

In [ ]:
# ── Cell 5: Formation energy + convex hull ────────────────────────────

from pymatgen.core import Composition
from hullgap.hull import (
    element_fraction,
    formation_energy_per_atom,
    lower_convex_hull_2d,
    hull_energy_at_x,
)

# Only keep rows that relaxed successfully
df_ok = df[~df.e_total_eV.isna()].copy()

# Mole fraction of element B
df_ok["x_B"] = df_ok["formula"].apply(
    lambda f: element_fraction(Composition(f), ELEMENT_B)
)

# Formation energy per atom
def _calc_eform(row):
    comp = Composition(row["formula"]) * (row["n_atoms"] / Composition(row["formula"]).num_atoms)
    return formation_energy_per_atom(row["e_total_eV"], comp, refs)

df_ok["e_form_eV_atom"] = df_ok.apply(_calc_eform, axis=1)

# Build lower convex hull including (0, 0) and (1, 0) anchors
anchors = np.array([[0.0, 0.0], [1.0, 0.0]])
candidate_pts = np.column_stack([df_ok.x_B.values, df_ok.e_form_eV_atom.values])
hull = lower_convex_hull_2d(np.vstack([anchors, candidate_pts]))

# Energy above hull
df_ok["e_above_hull_eV_atom"] = df_ok.apply(
    lambda r: r["e_form_eV_atom"] - hull_energy_at_x(hull, r["x_B"]),
    axis=1,
)
df_ok["on_hull"] = df_ok["e_above_hull_eV_atom"] <= 0.025

n_stable = df_ok.on_hull.sum()
print(f"\n{len(df_ok)} candidates scored")
print(f"  On / near hull (< 25 meV): {n_stable}")
print(f"  Above hull:                {len(df_ok) - n_stable}")
print()
df_ok.sort_values("e_above_hull_eV_atom").head(10)

In [ ]:
# ── Cell 6: Plot convex hull ──────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

fig, ax = plt.subplots(figsize=(10, 6))

above = df_ok[~df_ok.on_hull]
on = df_ok[df_ok.on_hull]

# Sort hull vertices by x for proper line drawing
hull_sorted = hull[np.argsort(hull[:, 0])]

# Filled hull region (hull shape down to y=0)
hull_fill_x = np.concatenate([[hull_sorted[0, 0]], hull_sorted[:, 0], [hull_sorted[-1, 0]]])
hull_fill_y = np.concatenate([[0.0], hull_sorted[:, 1], [0.0]])
ax.fill(hull_fill_x, hull_fill_y, color="#289ff0", alpha=0.08, zorder=1)

# Hull boundary line
ax.plot(hull_sorted[:, 0], hull_sorted[:, 1], "-", color="#289ff0", lw=2.5, zorder=5,
        label="Convex hull", solid_capstyle="round")

# Candidates above hull
if len(above) > 0:
    ax.scatter(
        above.x_B, above.e_form_eV_atom,
        c="#94a3b8", s=55, alpha=0.6, edgecolors="white", linewidths=0.5,
        label=f"Above hull ({len(above)})",
        zorder=6,
    )

# Candidates on / near hull
if len(on) > 0:
    ax.scatter(
        on.x_B, on.e_form_eV_atom,
        c="#289ff0", s=100, edgecolors="white", linewidths=1.0,
        label=f"On / near hull ({len(on)})",
        zorder=7,
    )

# Elemental endpoints
ax.scatter([0, 1], [0, 0], marker="D", c="#289ff0", s=80, edgecolors="white",
           linewidths=1.0, zorder=8)
ax.annotate(ELEMENT_A, (0, 0), textcoords="offset points", xytext=(-14, -16),
            fontsize=10, fontweight="bold", color="#1e3a5f")
ax.annotate(ELEMENT_B, (1, 0), textcoords="offset points", xytext=(6, -16),
            fontsize=10, fontweight="bold", color="#1e3a5f")

# Annotate on-hull formulas (avoid overlap with staggered offsets)
for j, (_, row) in enumerate(on.iterrows()):
    y_off = 10 if j % 2 == 0 else -16
    ax.annotate(
        row["formula"],
        (row["x_B"], row["e_form_eV_atom"]),
        textcoords="offset points", xytext=(6, y_off),
        fontsize=8.5, color="#1e3a5f", fontweight="semibold",
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
    )

# Drop lines from above-hull points down to the hull
for _, row in above.iterrows():
    e_hull = hull_energy_at_x(hull_sorted, row["x_B"])
    ax.plot([row["x_B"], row["x_B"]], [row["e_form_eV_atom"], e_hull],
            ls=":", lw=0.7, color="#94a3b8", alpha=0.5, zorder=2)

# Reference line at E_form = 0
ax.axhline(0, color="#cbd5e1", lw=1.0, ls="--", zorder=3)

# Axis styling
ax.set_xlabel(f"x({ELEMENT_B})  —  mole fraction of {ELEMENT_B}", fontsize=12)
ax.set_ylabel("Formation energy (eV/atom)", fontsize=12)
ax.set_title(f"{ELEMENT_A}\u2013{ELEMENT_B}  convex hull   (MatterGen + MatterSim)",
             fontsize=14, fontweight="bold")
ax.set_xlim(-0.05, 1.05)
y_min = min(df_ok.e_form_eV_atom.min(), hull_sorted[:, 1].min())
ax.set_ylim(y_min * 1.25, max(0.15, df_ok.e_form_eV_atom.max() * 1.1 + 0.05))
ax.legend(fontsize=9.5, loc="upper right", framealpha=0.9)
ax.grid(True, alpha=0.15)

fig.tight_layout()
plt.savefig(str(RESULTS_CSV).replace(".csv", ".png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {str(RESULTS_CSV).replace('.csv', '.png')}")

In [ ]:
# ── Cell 7: Save results ─────────────────────────────────────────────

df_ok.to_csv(RESULTS_CSV, index=False)
print(f"Results saved to {RESULTS_CSV}")

# Save relaxed structures as CIF
relaxed_dir = OUTPUT_DIR / "relaxed"
relaxed_dir.mkdir(parents=True, exist_ok=True)

from pymatgen.io.ase import AseAtomsAdaptor

saved = 0
for i, struct in enumerate(structures):
    if i in df_ok.idx.values:
        cif_path = relaxed_dir / f"{SYSTEM}_{i:03d}_{struct.composition.reduced_formula}.cif"
        struct.to(filename=str(cif_path))
        saved += 1

print(f"Saved {saved} relaxed CIF files to {relaxed_dir}")